In [ ]:
# Install all required libraries
!pip install rank_bm25 sentence-transformers chromadb groq wikipedia-api langchain langchain-groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install wikipedia -q
import wikipedia
print("wikipedia imported successfully")

  Preparing metadata (setup.py) ... done
wikipedia imported successfully


In [ ]:
import os
from google.colab import userdata

# Load Groq API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Core imports
import wikipedia
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from groq import Groq
import numpy as np

print("All imports successful.")

All imports successful.


In [ ]:
# Topics to fetch
topics = [
    "Retrieval-augmented generation",
    "Large language model",
    "Transformer (machine learning model)",
    "BERT (language model)",
    "Attention mechanism (artificial intelligence)"
]

raw_docs = []

for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        raw_docs.append({
            "title": page.title,
            "content": page.content
        })
        print(f"Fetched: {page.title}")
    except Exception as e:
        print(f"Failed: {topic} — {e}")

print(f"\nTotal articles fetched: {len(raw_docs)}")

Fetched: Retrieval-augmented generation
Fetched: Large language model
Fetched: Transformer (deep learning)
Fetched: BERT (language model)
Failed: Attention mechanism (artificial intelligence) — Page id "Attention mechanism (artificial intelligence)" does not match any pages. Try another id!

Total articles fetched: 4


In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits a long text into smaller overlapping chunks by word count.

    chunk_size=500: each chunk has 500 words
    overlap=50: next chunk starts 50 words before previous chunk ended
    — this prevents context from being cut off at boundaries
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward by 450 words, not 500

    return chunks


all_chunks = []
chunk_metadata = []

for doc in raw_docs:
    chunks = chunk_text(doc["content"], chunk_size=500, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "source": doc["title"],  # which article this chunk came from
            "chunk_index": i         # position of this chunk within that article
        })

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 51


In [ ]:
# BM25 requires tokenized input (list of word lists, not raw strings)
tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]

bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks.")

BM25 index built over 51 chunks.


In [ ]:
# Embedding function using all-MiniLM-L6-v2
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# In-memory ChromaDB client — no disk storage needed for this notebook
client = chromadb.Client()
collection = client.create_collection(
    name="week5_hybrid",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine similarity for sentence embeddings
)

# Add chunks in batches to avoid memory spike
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids = [str(j) for j in range(i, i+len(batch_chunks))]
    batch_meta = chunk_metadata[i:i+len(batch_chunks)]

    collection.add(
        documents=batch_chunks,
        ids=batch_ids,
        metadatas=batch_meta
    )

print(f"ChromaDB collection built: {collection.count()} chunks indexed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB collection built: 51 chunks indexed.


In [ ]:
def bm25_search(query, top_k=10):
    """
    Returns top_k chunks ranked by BM25 keyword score.

    Steps:
    1. Tokenize the query the same way we tokenized chunks (lowercase + split)
    2. BM25 scores every chunk in the corpus against the query
    3. Sort by score descending, return top_k
    """
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    # argsort gives indices from lowest to highest — [::-1] reverses to highest first
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "text": all_chunks[idx],
            "score": scores[idx],
            "index": int(idx),
            "source": chunk_metadata[idx]["source"]
        })

    return results


# Quick test
test = bm25_search("what is attention mechanism", top_k=3)
for r in test:
    print(f"Score: {r['score']:.3f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

Score: 3.829 | Source: BERT (language model)
The small model aims to fool the large model. DeBERTa (2020) is a significant architectural variant, with disentangled attention. Its key idea is to t
---
Score: 3.635 | Source: BERT (language model)
smaller datasets to optimize its performance on specific tasks such as natural language inference and text classification, and sequence-to-sequence-ba
---
Score: 3.605 | Source: Transformer (deep learning)
and each layer in a transformer model has multiple attention heads. While each attention head attends to the tokens that are relevant to each token, m
---


In [ ]:
def semantic_search(query, top_k=10):
    """
    Returns top_k chunks ranked by cosine similarity via ChromaDB.

    Unlike BM25, this understands meaning — so "how models understand context"
    can match chunks about "attention" even without exact word overlap.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "text": results["documents"][0][i],
            # ChromaDB returns distance (lower = more similar)
            # convert to similarity: 1 - distance
            "score": 1 - results["distances"][0][i],
            "index": int(results["ids"][0][i]),  # cast string ID back to int
            "source": results["metadatas"][0][i]["source"]
        })

    return output


# Quick test
test = semantic_search("how does self-attention work", top_k=3)
for r in test:
    print(f"Score: {r['score']:.3f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

Score: 0.467 | Source: Transformer (deep learning)
and each layer in a transformer model has multiple attention heads. While each attention head attends to the tokens that are relevant to each token, m
---
Score: 0.443 | Source: Transformer (deep learning)
an order of magnitude fewer parameters than LSTMs. One of its authors, Jakob Uszkoreit, suspected that attention without recurrence would be sufficien
---
Score: 0.426 | Source: Transformer (deep learning)
W V {\displaystyle W^{K},W^{V}} , thus: MultiQueryAttention ( Q , K , V ) = Concat i ∈ [ n heads ] ( Attention ( X W i Q , X W K , X W V ) ) W O {\dis
---


In [ ]:
def reciprocal_rank_fusion(bm25_results, semantic_results, k=60):
    """
    Merges two ranked lists into one using RRF.

    Formula: score = 1 / (k + rank) for each list, summed per document.

    Why k=60: it dampens the impact of top ranks — rank 1 vs rank 2
    difference is small, preventing one list from dominating the other.
    A chunk appearing in both lists gets two contributions — so it naturally
    rises to the top.
    """
    rrf_scores = {}   # chunk index → combined RRF score
    chunk_lookup = {} # chunk index → chunk data (text, source etc.)

    # Process BM25 results
    for rank, result in enumerate(bm25_results):
        idx = result["index"]
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
        chunk_lookup[idx] = result

    # Process semantic results
    for rank, result in enumerate(semantic_results):
        idx = result["index"]
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
        chunk_lookup[idx] = result

    # Sort by combined RRF score descending
    sorted_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    fused_results = []
    for idx in sorted_indices:
        result = chunk_lookup[idx].copy()
        result["rrf_score"] = rrf_scores[idx]
        fused_results.append(result)

    return fused_results


def hybrid_search(query, top_k=10):
    bm25_results = bm25_search(query, top_k=top_k)
    semantic_results = semantic_search(query, top_k=top_k)
    fused = reciprocal_rank_fusion(bm25_results, semantic_results)
    return fused[:top_k]


# Quick test
test = hybrid_search("BERT pretraining objective", top_k=3)
for r in test:
    print(f"RRF Score: {r['rrf_score']:.4f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

RRF Score: 0.0164 | Source: BERT (language model)
if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta
---
RRF Score: 0.0164 | Source: BERT (language model)
smaller datasets to optimize its performance on specific tasks such as natural language inference and text classification, and sequence-to-sequence-ba
---
RRF Score: 0.0161 | Source: Transformer (deep learning)
difficulty in converging. In the original paper, the authors recommended using learning rate warmup. That is, the learning rate should linearly scale 
---


In [ ]:
# Cross-encoder: sees query + document together — more accurate than bi-encoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [ ]:
def rerank(query, candidates, top_k=5):
    """
    Takes hybrid search results and reranks them using cross-encoder.

    Cross-encoder sees (query + document) together as one input — this is
    why it's more accurate than bi-encoder. It can model direct interaction
    between query words and document words, not just vector similarity.

    retrieve_k should be larger than final top_k — cast a wide net first,
    then let the reranker pick the best ones precisely.
    """
    if not candidates:
        return []

    # Create (query, document) pairs — cross-encoder scores each pair
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    # Attach reranker score to each candidate
    for i, candidate in enumerate(candidates):
        candidate["rerank_score"] = float(scores[i])

    # Sort by reranker score descending
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]


def hybrid_search_with_rerank(query, retrieve_k=20, final_k=5):
    """
    Full pipeline:
    Step 1 — Hybrid search retrieves top retrieve_k (wide net, fast)
    Step 2 — Reranker picks best final_k from those (precise, slow)

    retrieve_k=20, final_k=5 is a standard production ratio.
    """
    candidates = hybrid_search(query, top_k=retrieve_k)
    reranked = rerank(query, candidates, top_k=final_k)
    return reranked


# Quick test
test = hybrid_search_with_rerank("how does BERT use masked language modeling", retrieve_k=20, final_k=3)
for r in test:
    print(f"Rerank Score: {r['rerank_score']:.4f} | Source: {r['source']}")
    print(r['text'][:200])
    print("---")

Rerank Score: 5.9422 | Source: BERT (language model)
layer, translating a one-hot vector into a dense vector based on its token type. Position: The position embeddings are based on a token's position in the sequence. BERT uses absolute position embeddin
---
Rerank Score: 5.5520 | Source: BERT (language model)
if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for tasks like question answering or document classifica
---
Rerank Score: 5.5412 | Source: BERT (language model)
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-
---


In [ ]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_answer(query, context_chunks, model="llama-3.1-8b-instant"):
    """
    Takes query + top reranked chunks → generates a grounded answer via Groq.

    temperature=0.1 — keeps the answer factual, not creative.
    The prompt explicitly tells the model to only use the provided context
    — this is what prevents hallucination in RAG systems.
    """
    context = "\n\n".join([
        f"[Source: {c['source']}]\n{c['text']}"
        for c in context_chunks
    ])

    prompt = f"""You are a precise question-answering assistant.
Answer the question using ONLY the context provided below.
If the context does not contain the answer, say "I don't know based on the provided context."

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512
    )

    return response.choices[0].message.content


# End-to-end test
query = "What is the role of attention mechanism in transformers?"
context = hybrid_search_with_rerank(query, retrieve_k=20, final_k=5)
answer = generate_answer(query, context)

print(f"QUERY: {query}\n")
print(f"ANSWER:\n{answer}")

QUERY: What is the role of attention mechanism in transformers?

ANSWER:
The attention mechanism in transformers enables the model to process relationships between all elements in a sequence simultaneously, regardless of their distance from each other. It calculates "soft" weights for each token, more precisely for its embedding, by using multiple attention heads, each with its own "relevance" for calculating its own soft weights. This allows the model to capture more complex and long-range dependencies in deeper layers and to attend to the tokens that are relevant to each token for different definitions of "relevance".


In [ ]:
def compare_retrieval_methods(query, top_k=3):
    """
    Runs all three methods on the same query and prints results side by side.

    Purpose: visually see where each method wins and loses.
    - BM25 wins on exact keyword matches
    - Semantic wins on conceptual/paraphrased queries
    - Hybrid + Rerank wins on most queries by combining both
    """
    print(f"QUERY: {query}")
    print("=" * 70)

    # BM25
    print("\n[BM25 — Keyword Only]")
    bm25_results = bm25_search(query, top_k=top_k)
    for i, r in enumerate(bm25_results):
        print(f"  Rank {i+1} | Score: {r['score']:.3f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    # Semantic
    print("\n[Semantic — Embedding Only]")
    sem_results = semantic_search(query, top_k=top_k)
    for i, r in enumerate(sem_results):
        print(f"  Rank {i+1} | Score: {r['score']:.3f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    # Hybrid + Rerank
    print("\n[Hybrid + Rerank — Full Pipeline]")
    hybrid_results = hybrid_search_with_rerank(query, retrieve_k=20, final_k=top_k)
    for i, r in enumerate(hybrid_results):
        print(f"  Rank {i+1} | Rerank Score: {r['rerank_score']:.4f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    print("\n" + "=" * 70)


# Three queries — each designed to expose a different method's weakness
compare_retrieval_methods("BERT masked language model pretraining")  # exact term → BM25 should win
compare_retrieval_methods("how do neural networks understand context") # conceptual → semantic should win
compare_retrieval_methods("transformer self-attention mechanism explanation") # both → hybrid should win

QUERY: BERT masked language model pretraining

[BM25 — Keyword Only]
  Rank 1 | Score: 9.604 | Source: Transformer (deep learning)
  difficulty in converging. In the original paper, the authors recommended using learning rate warmup. That is, the learning rate should linearly scale ...
  Rank 2 | Score: 6.693 | Source: Large language model
  compound model is then fine-tuned on an image-text dataset. This basic construction can be applied with more sophistication to improve the model. The ...
  Rank 3 | Score: 6.651 | Source: BERT (language model)
  if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta...

[Semantic — Embedding Only]
  Rank 1 | Score: 0.716 | Source: BERT (language model)
  if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta...
  Rank 2 | Score: 0.680 | Source: BERT (language 

DAY 1A — HYBRID SEARCH + RERANKING
Advanced RAG | Month 1, Week 5

WHAT IS THIS NOTEBOOK?
This notebook implements an advanced RAG retrieval pipeline that goes beyond basic semantic search. It combines three retrieval strategies and benchmarks them against each other on the same queries.

WHAT WE BUILT:
1. BM25 Search — keyword-based retrieval using term frequency and inverse document frequency. No embeddings, purely statistical.

2. Semantic Search — embedding-based retrieval using all-MiniLM-L6-v2 (384-dim) stored in ChromaDB with HNSW index and cosine similarity.

3. Hybrid Search — combined BM25 + Semantic using Reciprocal Rank Fusion (RRF, k=60) to merge ranked lists without worrying about score scale differences.

4. Reranking — cross-encoder (ms-marco-MiniLM-L-6-v2) that sees query + document together for precise final ranking. Two-stage pipeline: retrieve top-20, rerank to top-5.

5. Answer Generation — Groq (llama-3.1-8b-instant) generates grounded answers from the top reranked chunks only.

CORPUS:
4 Wikipedia articles — RAG, LLMs, Transformers, BERT
Chunked into 51 chunks (500 words, 50-word overlap)

WHAT WE ACHIEVED:
- Hybrid + Rerank outperformed both BM25 and semantic alone on exact and mixed queries
- Identified that vague queries fail at retrieval level — no ranking method fixes a bad query (motivation for Day 1B)
- Negative reranker scores correctly flagged low-confidence retrievals — reranker acts as a quality gate, not just a sorter
- Observed that noisy corpus (Wikipedia math markup) hurts all methods equally — preprocessing is a real concern

KEY NUMBERS:
BM25 top score:      9.604  (exact keyword match)
Semantic top score:  0.716  (cosine similarity)
Reranker top score:  5.94   (cross-encoder confidence)
Reranker low score: -2.17   (actively flagged as irrelevant)



In [19]:
# Verify all Day 1A components are still loaded
print(f"Chunks: {len(all_chunks)}")
print(f"BM25 index: {bm25}")
print(f"ChromaDB collection: {collection.count()} docs")
print(f"Reranker: {reranker}")
print(f"Groq client: {groq_client}")

Chunks: 51
BM25 index: <rank_bm25.BM25Okapi object at 0x7906abb64500>
ChromaDB collection: 51 docs
Reranker: CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'BertForSequenceClassification'})
)
Groq client: <groq.Groq object at 0x7906a8592e70>


In [20]:
def query_expansion(query, n_variants=3):
    """
    Sends the original query to Groq and asks it to generate
    n_variants different phrasings of the same question.

    Why: different phrasings hit different chunks in retrieval.
    The union of all results has higher recall than any single query.

    We ask for JSON output so we can parse variants cleanly
    without any text splitting hacks.
    """
    prompt = f"""You are a search query expansion expert.
Given a question, generate {n_variants} different ways to ask the same question.
Each variant should use different vocabulary and phrasing but mean the same thing.

Return ONLY a JSON object in this exact format, no explanation:
{{"variants": ["variant1", "variant2", "variant3"]}}

Original question: {query}"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,  # slightly creative to get diverse phrasings
        max_tokens=200
    )

    raw = response.choices[0].message.content.strip()

    # Parse JSON response
    import json
    parsed = json.loads(raw)
    variants = parsed["variants"]

    # Always include original query in the list
    all_queries = [query] + variants

    return all_queries


# Test it
query = "how does BERT understand language"
expanded = query_expansion(query, n_variants=3)

print(f"Original: {query}\n")
print("Expanded queries:")
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

Original: how does BERT understand language

Expanded queries:
  1. how does BERT understand language
  2. What is the linguistic processing mechanism of BERT?
  3. How does BERT's architecture enable language comprehension?
  4. What are the key linguistic features that BERT uses to analyze language?


In [21]:
def expanded_retrieval(query, n_variants=3, top_k=5):
    """
    Runs hybrid search on the original query + all expanded variants.
    Merges results using RRF across all query retrievals.

    Why RRF again here: same reason as before — we can't average
    scores across different queries. Rank positions are always comparable.

    Steps:
    1. Generate query variants
    2. Run hybrid search on each variant separately
    3. Merge all results using RRF
    4. Return top_k deduplicated chunks
    """
    all_queries = query_expansion(query, n_variants=n_variants)

    print(f"Running retrieval on {len(all_queries)} queries...")

    # Collect all per-query results
    all_ranked_lists = []
    for q in all_queries:
        results = hybrid_search(q, top_k=top_k)
        all_ranked_lists.append(results)
        print(f"  Query: '{q[:60]}' → {len(results)} chunks retrieved")

    # Merge all ranked lists using RRF
    rrf_scores = {}
    chunk_lookup = {}

    for ranked_list in all_ranked_lists:
        for rank, result in enumerate(ranked_list):
            idx = result["index"]
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank + 1)
            chunk_lookup[idx] = result

    # Sort by merged RRF score
    sorted_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    fused = []
    for idx in sorted_indices:
        result = chunk_lookup[idx].copy()
        result["expansion_rrf_score"] = rrf_scores[idx]
        fused.append(result)

    return fused[:top_k]


# Test it
query = "how does BERT understand language"
results = expanded_retrieval(query, n_variants=3, top_k=5)

print(f"\nTop results after expansion + fusion:")
for i, r in enumerate(results):
    print(f"  Rank {i+1} | Score: {r['expansion_rrf_score']:.4f} | Source: {r['source']}")
    print(f"  {r['text'][:150]}...")

Running retrieval on 4 queries...
  Query: 'how does BERT understand language' → 5 chunks retrieved
  Query: 'What is the mechanism behind BERT's language comprehension?' → 5 chunks retrieved
  Query: 'How does BERT process and interpret linguistic inputs?' → 5 chunks retrieved
  Query: 'Can you explain the language understanding capabilities of t' → 5 chunks retrieved

Top results after expansion + fusion:
  Rank 1 | Score: 0.0492 | Source: BERT (language model)
  smaller datasets to optimize its performance on specific tasks such as natural language inference and text classification, and sequence-to-sequence-ba...
  Rank 2 | Score: 0.0476 | Source: BERT (language model)
  then naively one would mask out all the tokens as "Today, I went to [MASK] [MASK] [MASK] ... [MASK] ." where the number of [MASK] is the length of the...
  Rank 3 | Score: 0.0474 | Source: BERT (language model)
  Bidirectional encoder representations from transformers (BERT) is a language model introduced in October

In [22]:
def hyde(query, top_k=5):
    """
    Hypothetical Document Embeddings.

    Instead of embedding the query directly, we:
    1. Ask the LLM to generate a fake but plausible answer paragraph
    2. Embed that paragraph instead of the query
    3. Search the vector store with that embedding

    Why: a generated answer paragraph lives in the same vector space
    as real document chunks — much closer than a short question would be.
    Even a factually wrong hypothetical retrieves the right chunks
    because the vocabulary and style match real documents.
    """

    # Step 1: Generate hypothetical answer
    prompt = f"""Write a short, factual paragraph (3-5 sentences) that directly answers
the following question. Write it as if it were extracted from a technical article.
Do not mention that this is hypothetical. Just write the answer paragraph.

Question: {query}

Paragraph:"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,  # low temp — we want factual, not creative
        max_tokens=200
    )

    hypothetical_doc = response.choices[0].message.content.strip()
    print(f"Hypothetical document generated:\n{hypothetical_doc}\n")

    # Step 2: Search using hypothetical doc as query (not original query)
    results = collection.query(
        query_texts=[hypothetical_doc],  # embed the paragraph, not the question
        n_results=top_k
    )

    # Step 3: Format results
    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
            "index": int(results["ids"][0][i]),
            "source": results["metadatas"][0][i]["source"]
        })

    return output, hypothetical_doc


# Test it
query = "how does masked language modeling work in BERT"
results, hypo_doc = hyde(query, top_k=5)

print("Top results via HyDE:")
for i, r in enumerate(results):
    print(f"  Rank {i+1} | Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"  {r['text'][:150]}...")

Hypothetical document generated:
Masked language modeling is a key component of the BERT (Bidirectional Encoder Representations from Transformers) architecture. In this approach, a portion of the input tokens are randomly replaced with a [MASK] token, which is then predicted by the model. The model is trained to predict the original token that was masked, based on the context provided by the surrounding tokens. This process encourages the model to learn contextual relationships between words and their meanings, enabling it to better capture nuances of language. By doing so, masked language modeling helps BERT to develop a robust understanding of language that can be leveraged for a wide range of natural language processing tasks.

Top results via HyDE:
  Rank 1 | Score: 0.8122 | Source: BERT (language model)
  if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta...
  Rank 2 | Score: 0.7862

In [23]:
# 20 questions across 4 types — 5 each
# Designed to stress-test each method's weakness

questions = [
    # Type 1 — Exact keyword (BM25 should win)
    "What is BM25 Okapi",
    "What is BERT masked language model",
    "What is retrieval augmented generation",
    "What is next sentence prediction in BERT",
    "What is multi-head attention",

    # Type 2 — Conceptual / paraphrased (Semantic / HyDE should win)
    "how do neural networks focus on important words",
    "why do transformers outperform recurrent networks",
    "how does a model learn from unlabeled text",
    "what makes large language models good at following instructions",
    "how do models handle long range dependencies",

    # Type 3 — Vague / short (Query Expansion should win)
    "BERT training",
    "attention",
    "RAG",
    "transformer layers",
    "language model evaluation",

    # Type 4 — Specific technical (Hybrid + Rerank should win)
    "how does positional encoding work in transformers",
    "what is the difference between encoder and decoder in transformers",
    "how does BERT handle out of vocabulary words",
    "what indexing strategies does FAISS use",
    "how does retrieval augmented generation reduce hallucinations",
]

print(f"Total benchmark questions: {len(questions)}")
print("\nQuestion types:")
print("  Q1-5:   Exact keyword")
print("  Q6-10:  Conceptual / paraphrased")
print("  Q11-15: Vague / short")
print("  Q16-20: Specific technical")

Total benchmark questions: 20

Question types:
  Q1-5:   Exact keyword
  Q6-10:  Conceptual / paraphrased
  Q11-15: Vague / short
  Q16-20: Specific technical


In [24]:
import time

def run_benchmark(questions):
    """
    Runs all 4 retrieval methods on every question.
    Records top 3 retrieved chunks per method per question.

    We store results in a list of dicts — one per question.
    This makes scoring and comparison easy in the next cell.

    We add a small sleep between questions to avoid hitting
    Groq rate limits — query expansion and HyDE each make
    an LLM call per question.
    """
    results = []

    for i, query in enumerate(questions):
        print(f"Q{i+1}/{len(questions)}: {query[:50]}...")

        entry = {"query": query, "methods": {}}

        # Method 1 — BM25
        entry["methods"]["bm25"] = bm25_search(query, top_k=3)

        # Method 2 — Semantic
        entry["methods"]["semantic"] = semantic_search(query, top_k=3)

        # Method 3 — Hybrid + Rerank
        entry["methods"]["hybrid_rerank"] = hybrid_search_with_rerank(
            query, retrieve_k=20, final_k=3
        )

        # Method 4 — Query Expansion + Hybrid
        try:
            expanded = expanded_retrieval(query, n_variants=3, top_k=3)
            entry["methods"]["query_expansion"] = expanded
        except Exception as e:
            print(f"  Query expansion failed: {e}")
            entry["methods"]["query_expansion"] = []

        # Method 5 — HyDE
        try:
            hyde_results, _ = hyde(query, top_k=3)
            entry["methods"]["hyde"] = hyde_results
        except Exception as e:
            print(f"  HyDE failed: {e}")
            entry["methods"]["hyde"] = []

        results.append(entry)

        # Sleep to avoid Groq rate limits
        time.sleep(2)
        print(f"  Done.\n")

    return results


print("Starting benchmark — this will take ~2-3 minutes due to rate limit sleeps...")
benchmark_results = run_benchmark(questions)
print("Benchmark complete.")

Starting benchmark — this will take ~2-3 minutes due to rate limit sleeps...
Q1/20: What is BM25 Okapi...
Running retrieval on 4 queries...
  Query: 'What is BM25 Okapi' → 3 chunks retrieved
  Query: 'What is the Okapi BM25 algorithm?' → 3 chunks retrieved
  Query: 'What is BM25, and how does it relate to the Okapi search mod' → 3 chunks retrieved
  Query: 'Can you explain the BM25 Okapi weighting formula?' → 3 chunks retrieved
Hypothetical document generated:
BM25 Okapi is a variant of the BM25 algorithm, a widely used ranking function in information retrieval. It was developed by Microsoft Research and is based on the Okapi BM25 algorithm, which was originally designed for searching large collections of documents. BM25 Okapi incorporates additional features, such as a more robust handling of query term frequency and a more accurate estimation of document length, to improve the ranking of search results. These enhancements enable BM25 Okapi to produce more accurate and relevant search

In [25]:
def score_benchmark(benchmark_results):
    """
    Manually scores each method's top-3 results per question.

    Scoring logic:
    - Hit@1: correct chunk at rank 1 → 2 points
    - Hit@3: correct chunk in top 3 → 1 point
    - Miss:  nothing relevant in top 3 → 0 points

    "Correct" = the retrieved chunk actually answers the question.
    We check this by looking at the source — if the source matches
    what we'd expect for that question, it's a hit.

    Expected sources per question type:
    - BERT questions → BERT (language model)
    - Transformer questions → Transformer (deep learning)
    - RAG questions → Retrieval-augmented generation
    - LLM questions → Large language model
    - Mixed → any relevant source
    """

    # Expected source per question
    expected_sources = [
        "Transformer (deep learning)",          # Q1  BM25 Okapi — not in corpus, any technical = ok
        "BERT (language model)",                # Q2
        "Retrieval-augmented generation",       # Q3
        "BERT (language model)",                # Q4
        "Transformer (deep learning)",          # Q5
        "Transformer (deep learning)",          # Q6
        "Transformer (deep learning)",          # Q7
        "BERT (language model)",                # Q8
        "Large language model",                 # Q9
        "Transformer (deep learning)",          # Q10
        "BERT (language model)",                # Q11
        "Transformer (deep learning)",          # Q12
        "Retrieval-augmented generation",       # Q13
        "Transformer (deep learning)",          # Q14
        "Large language model",                 # Q15
        "Transformer (deep learning)",          # Q16
        "Transformer (deep learning)",          # Q17
        "BERT (language model)",                # Q18
        "Retrieval-augmented generation",       # Q19 FAISS — not in corpus
        "Retrieval-augmented generation",       # Q20
    ]

    methods = ["bm25", "semantic", "hybrid_rerank", "query_expansion", "hyde"]
    scores = {m: {"hit1": 0, "hit3": 0, "total": 0} for m in methods}

    for i, entry in enumerate(benchmark_results):
        expected = expected_sources[i]

        for method in methods:
            results = entry["methods"].get(method, [])
            if not results:
                continue

            # Check Hit@1
            if results[0]["source"] == expected:
                scores[method]["hit1"] += 1
                scores[method]["total"] += 2

            # Check Hit@3 (excluding rank 1 to avoid double counting)
            elif any(r["source"] == expected for r in results[1:3]):
                scores[method]["hit3"] += 1
                scores[method]["total"] += 1

    return scores


scores = score_benchmark(benchmark_results)

print("=" * 55)
print(f"{'Method':<20} {'Hit@1':>6} {'Hit@3':>6} {'Total/40':>9}")
print("=" * 55)
for method, s in scores.items():
    print(f"{method:<20} {s['hit1']:>6} {s['hit3']:>6} {s['total']:>9}")
print("=" * 55)
print("Max possible score: 40 (20 questions × 2 points)")

Method                Hit@1  Hit@3  Total/40
bm25                     15      4        34
semantic                 15      1        31
hybrid_rerank            18      0        36
query_expansion          16      2        34
hyde                     18      0        36
Max possible score: 40 (20 questions × 2 points)


In [27]:
print("""
BENCHMARK ANALYSIS — WEEK 5 DAY 1B
====================================

SCORES SUMMARY:
  Hybrid + Rerank   36/40  ← joint winner
  HyDE              36/40  ← joint winner
  BM25              34/40
  Query Expansion   34/40
  Semantic          31/40

KEY OBSERVATIONS:

1. HYBRID + RERANK (36/40)
   - Consistently strong across all query types
   - Never catastrophically failed on any question
   - Best default choice for production RAG

2. HyDE (36/40)
   - Surprisingly strong — tied with hybrid+rerank
   - Won on conceptual queries where semantic alone scored lower
   - BUT completely failed Q12 ('attention') and Q13 ('RAG')
     — single word queries broke the hypothetical generation
     — HyDE generated neuroscience and DNA biology paragraphs
   - Rule: never use HyDE on queries under 4 words

3. BM25 (34/40)
   - Performed better than expected — 15 Hit@1
   - Our corpus is technical with exact terminology
   - Would degrade badly on paraphrased or conversational queries

4. QUERY EXPANSION (34/40)
   - Same score as BM25 but for different reasons
   - Q12 'attention' expansion went off-track completely:
     generated 'being mindful of something' as a variant
   - Works well on medium-length queries (Q2-Q11)
   - Adds latency — 4 LLM calls per query vs 0 for BM25

5. SEMANTIC (31/40)
   - Lowest scorer — hurt by exact technical terms in corpus
   - BM25 outperformed it on keyword-heavy questions
   - Semantic alone is not enough for technical RAG

WHAT TO WRITE IN NOTEBOOK:

- No single method wins everything
- Hybrid + Rerank is the safest default
- HyDE adds value on conceptual queries but breaks on short queries
- Query Expansion helps on vague queries but adds cost and latency
- Production decision: start with Hybrid+Rerank, add HyDE
  only for queries that are longer than 4 words
- Single word queries need query expansion BEFORE any retrieval
""")


BENCHMARK ANALYSIS — WEEK 5 DAY 1B

SCORES SUMMARY:
  Hybrid + Rerank   36/40  ← joint winner
  HyDE              36/40  ← joint winner  
  BM25              34/40
  Query Expansion   34/40
  Semantic          31/40

KEY OBSERVATIONS:

1. HYBRID + RERANK (36/40)
   - Consistently strong across all query types
   - Never catastrophically failed on any question
   - Best default choice for production RAG

2. HyDE (36/40)
   - Surprisingly strong — tied with hybrid+rerank
   - Won on conceptual queries where semantic alone scored lower
   - BUT completely failed Q12 ('attention') and Q13 ('RAG')
     — single word queries broke the hypothetical generation
     — HyDE generated neuroscience and DNA biology paragraphs
   - Rule: never use HyDE on queries under 4 words

3. BM25 (34/40)
   - Performed better than expected — 15 Hit@1
   - Our corpus is technical with exact terminology
   - Would degrade badly on paraphrased or conversational queries

4. QUERY EXPANSION (34/40)
   - Same score

# Day 1B — Query Expansion, HyDE & Retrieval Benchmark
## Week 5 — Advanced RAG Techniques | Month 1

---

## What This Notebook Covers
Building on Day 1A (Hybrid Search + Reranking), Day 1B focuses on
fixing the query itself before retrieval happens. We implement two
advanced query-side techniques and benchmark all methods head-to-head
on 20 questions.

---

## Techniques Implemented

### 1. Query Expansion
Generate multiple rephrasings of the original query using an LLM,
run retrieval on each variant separately, then merge all results
using Reciprocal Rank Fusion (RRF).

**Why:** Different phrasings hit different chunks. The union of all
retrievals has higher recall than any single query alone.

**Cost:** Extra LLM calls per query (1 call → 4 queries)

### 2. HyDE (Hypothetical Document Embeddings)
Instead of embedding the query directly, ask the LLM to generate
a hypothetical answer paragraph, embed that paragraph, and search
with it.

**Why:** A generated answer paragraph lives in the same vector space
as real document chunks — much closer than a short question would be.

**Cost:** One extra LLM call per query. Breaks on queries under 4 words.

---

## Benchmark Setup
- **Corpus:** 4 Wikipedia articles → 51 chunks (same as Day 1A)
- **Questions:** 20 questions across 4 types
  - Q1–5: Exact keyword queries
  - Q6–10: Conceptual / paraphrased queries
  - Q11–15: Vague / short queries
  - Q16–20: Specific technical queries
- **Scoring:** Hit@1 = 2pts | Hit@3 = 1pt | Miss = 0pt | Max = 40pts
- **Methods compared:** BM25, Semantic, Hybrid+Rerank, Query Expansion, HyDE

---

## Results

| Method | Hit@1 | Hit@3 | Total/40 |
|---|---|---|---|
| Hybrid + Rerank | 18 | 0 | **36** |
| HyDE | 18 | 0 | **36** |
| BM25 | 15 | 4 | 34 |
| Query Expansion | 16 | 2 | 34 |
| Semantic | 15 | 1 | 31 |

---

## Key Findings

**Hybrid + Rerank (36/40)**
- Consistently strong across all 4 query types
- Never catastrophically failed on any question
- Best default choice for production RAG

**HyDE (36/40)**
- Tied with Hybrid+Rerank — surprisingly strong
- Won on conceptual queries where semantic alone underperformed
- Completely failed on single-word queries:
  - "attention" → generated a neuroscience paragraph
  - "RAG" → generated a DNA biology paragraph
- Rule: never use HyDE on queries under 4 words

**BM25 (34/40)**
- Performed better than expected on this technical corpus
- Exact terminology in Wikipedia articles favoured keyword matching
- Would degrade badly on conversational or paraphrased queries

**Query Expansion (34/40)**
- Worked well on medium-length queries (Q2–Q11)
- Failed on "attention" — LLM expanded it as cognitive attention,
  not ML attention
- Adds latency: 4 LLM calls per query vs 0 for BM25

**Semantic Alone (31/40)**
- Lowest scorer — hurt by exact technical terminology in corpus
- BM25 outperformed it on keyword-heavy questions
- Semantic alone is not sufficient for technical RAG

---

## Failure Cases Worth Noting

| Query | Method | What Went Wrong |
|---|---|---|
| "attention" | HyDE | Generated neuroscience paragraph |
| "RAG" | HyDE | Generated DNA biology paragraph |
| "attention" | Query Expansion | Expanded to "being mindful of something" |
| "RAG" | Query Expansion | Interpreted as evaluation metrics |

**Root cause:** Single-word and 2-word queries are too ambiguous for
LLM-based techniques. The LLM has no context to anchor on.

---

## Production Decision Framework

Query length < 4 words?
→ Apply Query Expansion first, then Hybrid+Rerank
Query is conceptual / paraphrased?
→ Use HyDE + Hybrid+Rerank
Query has exact technical terms?
→ Hybrid+Rerank alone is sufficient
Default for everything else?
→ Hybrid+Rerank

---

## Stack Used
- **LLM:** Groq — llama-3.1-8b-instant
- **Embeddings:** all-MiniLM-L6-v2 (384-dim)
- **Vector Store:** ChromaDB (HNSW, cosine)
- **Keyword Search:** rank-bm25 (BM25Okapi)
- **Reranker:** cross-encoder/ms-marco-MiniLM-L-6-v2
- **Corpus:** Wikipedia (RAG, LLMs, Transformers, BERT) → 51 chunks

---